# T4 final measurements — run top to bottom

**Runtime → Change runtime type → T4 GPU**, then Run all. Total ~15 phút.

Notebook này chạy **hai phép đo cuối còn thiếu** của đồ án, cộng một phép tái lập bonus:

| # | Phép đo | Vì sao cần |
|---|---|---|
| 1 | Test suite trên GPU thật | cổng vào — không xanh thì mọi số phía dưới vô nghĩa |
| 2 | **Parity gate trên 1231 frame CDnet thật** | trước giờ mới chứng minh trên frame tổng hợp |
| 3 | **FPS trên footage 1080p thật, 4 độ phân giải** | trước giờ timing 1080p đo trên frame tổng hợp |
| 4 | (bonus) Tái lập F1 0.9843 trên máy thứ hai | record hiện tại đo trên máy ARM |

Khi chạy xong, **copy toàn bộ output của cell cuối** gửi lại — nó gom kết quả của mọi cell.

In [ ]:
# ── 1. Setup: GPU, repo, deps, test suite ────────────────────────────────────
import os, subprocess, sys, json, time
T0 = time.time()
RESULTS = {}

print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],capture_output=True,text=True).stdout.strip() or "NO GPU — fix runtime type first!")

if os.path.isdir("/content/project"): subprocess.run(["rm","-rf","/content/project"])
subprocess.run(["git","clone","-q","--depth","1",
  "https://github.com/Sn-cpp/Adaptive-Gaussian-Mixture-Model.git","/content/project"],check=True)
os.chdir("/content/project")
RESULTS["commit"] = subprocess.run(["git","log","--oneline","-1"],capture_output=True,text=True).stdout.strip()
print(RESULTS["commit"])

subprocess.run([sys.executable,"-m","pip","-q","install","numba<0.62"],capture_output=True)
import cv2, numba
RESULTS["env"] = f"python {sys.version.split()[0]} | cv2 {cv2.__version__} | numba {numba.__version__}"
print(RESULTS["env"])

r = subprocess.run([sys.executable,"-m","pytest","tests/","-q","--no-header","-p","no:cacheprovider"],
                   capture_output=True,text=True)
line = [l for l in r.stdout.splitlines() if "passed" in l or "failed" in l][-1]
RESULTS["tests"] = line
print("tests:", line)
assert "failed" not in line, "STOP — suite is red, numbers below would be meaningless"

In [ ]:
# ── 2. Dataset from the team mirror ──────────────────────────────────────────
import urllib.request, zipfile, hashlib, socket
socket.setdefaulttimeout(60)
url = ("https://huggingface.co/datasets/haiduonghuynhle/"
       "changedetection-2012-highway/resolve/main/highway.zip")
urllib.request.urlretrieve(url, "highway.zip")
sha = hashlib.sha256(open("highway.zip","rb").read()).hexdigest()
print("sha256:", sha)
assert sha == "44878b640a1c8c9e9a5fb5c447c596c7d21ccdf032dfc3f869313375fb1b210c", \
    "dataset changed since it was recorded — do not quote results against it"
with zipfile.ZipFile("highway.zip") as z: z.extractall(".")
n_in  = len(os.listdir("highway/input"))
n_gt  = len(os.listdir("highway/groundtruth"))
RESULTS["dataset"] = f"highway: {n_in} input, {n_gt} gt, sha OK"
print(RESULTS["dataset"]); assert n_in == 1700 and n_gt == 1700

In [ ]:
# ── 3. THE PARITY GATE — shipping mask+composite on the real 1231 frames ────
# v2 (the submitted version) and v1, each against the Numba host reference,
# frame by frame over the full scored window. This is the check that until now
# only ever ran on synthetic frames.
for model in ("cuda_v2", "cuda_v1"):
    print(f"\n=== parity {model} vs numba, real CDnet frames ===")
    r = subprocess.run([sys.executable,"eval_highway.py","--model",model,
                        "--parity-vs","numba","--colorspace","ycrcb"],
                       capture_output=True,text=True,
                       env={**os.environ,"HIGHWAY_DIR":"highway"})
    print(r.stdout[-1500:] or r.stderr[-700:])
    # exit 0 covers both verdicts: IDENTICAL, and FLOAT-BOUNDARY (every
    # flipped pixel individually proven to sit on the 0.5 threshold with
    # |bg_prob_a - bg_prob_b| < 1e-4 — FMA contraction, not a logic bug).
    verdict = next((l for l in r.stdout.splitlines() if l.strip().startswith("VERDICT:")), "no verdict line")
    RESULTS[f"parity_{model}"] = verdict.strip()
    assert r.returncode == 0, f"{model} parity gate FAILED"

In [ ]:
# ── 4. FPS on real Full-HD footage, four resolutions ─────────────────────────
# One fixed-camera 1080p traffic clip (Pexels 4791721, free licence), decoded
# once, downscaled to every tier — same scene, only the pixel count changes.
r = subprocess.run([sys.executable,"benchmarks/bench_realfootage.py","--frames","160"],
                   capture_output=True,text=True)
print(r.stdout[-1600:] or r.stderr[-1200:])
RESULTS["realfootage"] = r.stdout[r.stdout.find("real footage"):] if "real footage" in r.stdout else "ERROR"
assert r.returncode == 0

In [ ]:
# ── 5. (bonus) Reproduce F1 0.9843 on this second machine ────────────────────
# The committed record (benchmarks/records/eval_highway_full.txt) was measured
# on an ARM host. Same command here on x86: the digits should not move, and if
# they do that is a finding.
r = subprocess.run([sys.executable,"eval_highway.py","--colorspace","ycrcb"],
                   capture_output=True,text=True,
                   env={**os.environ,"HIGHWAY_DIR":"highway"})
tail = r.stdout[r.stdout.find("highway 470"):]
print(tail[:900])
top = [l for l in tail.splitlines() if "median5 + fill" in l and "close" not in l]
RESULTS["f1_repro"] = top[0].strip() if top else "ERROR"

In [ ]:
# ── 6. COPY EVERYTHING BELOW THIS LINE ───────────────────────────────────────
print("="*72)
print("T4 FINAL MEASUREMENTS — paste this whole block back")
print("="*72)
for k, v in RESULTS.items():
    print(f"\n[{k}]")
    print(v if isinstance(v,str) else json.dumps(v))
print(f"\ntotal wall time: {(time.time()-T0)/60:.1f} min")
print("="*72)